# RLHF: tayyor reward model (Skywork-Reward-V2) + RLOO, SFT v3 LoRA (Qwen2.5-1.5B) ustida

Reward model'ni o'zimiz o'qitmaymiz: Hugging Face'dagi tayyor **Skywork/Skywork-Reward-V2-Qwen3-1.7B** ishlatiladi
(reward model jadvalidagi 4-variant). U millionlab inglizcha juftliklarda o'qitilgan, lekin o'zbek tilini kam ko'rgan.

| | RLHF v2 | Bu notebook |
|---|---|---|
| Reward model | o'zimiz o'qitgan (Qwen2.5-1.5B + SFT v3, 1 939 juftlik) | tayyor Skywork-Reward-V2 (Qwen3-1.7B) |
| O'qitish | ~30 daqiqa | yo'q |
| Tokenizer | policy bilan bir xil | boshqa (Qwen3): javob matnga qaytarilib, qayta tokenlanadi |
| RL | RLOO, 4 ta javob, KL 0.1 | xuddi shunday |

**Boshlashdan oldin:** **GPU T4 x2**, **Internet -> On**, **Add Input** -> `sft-v3-adapters`.
**Save Version -> Save & Run All (Commit)**. Taxminiy vaqt: ~3-4 soat.

## 1. GPU borligini tekshirish

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Loyiha va base modelni yuklab olish

In [ ]:
%cd /kaggle/working
!rm -rf Post_Training
!GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/davronbek-malikov/Post_Training.git
%cd Post_Training
!pip -q install -U transformers peft bitsandbytes accelerate
!pip -q uninstall -y torchao  # Kaggle'dagi eski torchao peft bilan adapter yuklashni buzadi

from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-1.5B", local_dir="Qwen2.5-1.5B")
!ls -lh Qwen2.5-1.5B/*.safetensors

## 3. SFT v3 LoRA adapter (policy va reference shundan boshlanadi)

In [ ]:
import glob, os, shutil, subprocess, zipfile
target = "chat_ui/adapters/sft-lora-v3"
found = glob.glob("/kaggle/input/**/sft-lora-v3/adapter_config.json", recursive=True)
if not found:   # zip ochilmagan holda yuklangan bo'lsa
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        with zipfile.ZipFile(z) as f:
            names = [n for n in f.namelist() if "sft-lora-v3/" in n]
            if names:
                f.extractall("/kaggle/working/unzipped", members=names)
    found = glob.glob("/kaggle/working/unzipped/**/sft-lora-v3/adapter_config.json", recursive=True)
if found:
    shutil.copytree(os.path.dirname(found[0]), target, dirs_exist_ok=True)
    print("SFT v3 adapter topildi:", os.path.dirname(found[0]))
else:
    print("Adapter topilmadi: SFT v3 LoRA qaytadan o'qitiladi...")
    subprocess.run(["python", "sft_v2/train.py", "--method", "lora", "--data", "sft_v3/data",
                    "--name", "sft-lora-v3", "--epochs", "3", "--rank", "64"], check=True)
print(os.listdir(target))

## 4. Hakamni tekshirish (o'qitishsiz)

Skywork bizning 181 ta val juftlikda qanchalik to'g'ri? Bizning v2 reward model: **0.84**.
`o'zbekcha savollar` qatoriga e'tibor bering.

In [ ]:
!python rlhf/eval_judge.py --judge hf --model Skywork/Skywork-Reward-V2-Qwen3-1.7B

## 5. RLOO (Skywork hakam)

Hakam ballari har qadamda normallashtiriladi (o'rtacha 0, og'ish 1): Skywork'ning o'z shkalasi bor.
Kuzating: `RM ball` va `qoida bali` birga o'sishi kerak; `uzunlik` keskin o'zgarmasligi kerak.

In [ ]:
!python rlhf/train_ppo.py --algo rloo --group-size 4 --batch-size 8 --mini-batch 4 --kl-coef 0.1 \
    --judge hf --judge-model Skywork/Skywork-Reward-V2-Qwen3-1.7B \
    --name sft-lora-rlhf-skywork --rollouts rlhf_skywork_rollouts.jsonl
import os
assert os.path.exists("chat_ui/adapters/sft-lora-rlhf-skywork/adapter_config.json"), 'RLOO adapterni saqlamadi: yuqoridagi xatoni qarang'

In [ ]:
!rm -f /kaggle/working/sft-lora-rlhf-skywork_results.zip
!zip -q -r /kaggle/working/sft-lora-rlhf-skywork_results.zip chat_ui/adapters/sft-lora-rlhf-skywork rlhf/logs/rlhf_skywork_rollouts.jsonl
!ls -lh /kaggle/working/sft-lora-rlhf-skywork_results.zip

## Baholash: 68 ta test savoli

SFT v3: 41/68. DPO v2: 44/68. RLHF v1: 40/68.

In [ ]:
!python eval/run_eval.py --base Qwen2.5-1.5B --model sft-lora-rlhf-skywork | tail -12
!zip -q /kaggle/working/sft-lora-rlhf-skywork_results.zip eval/results/sft-lora-rlhf-skywork.json
assert os.path.exists("eval/results/sft-lora-rlhf-skywork.json"), 'Baholash ishlamadi: yuqoridagi xatoni qarang'

## Yuklab olish

**Output** -> `sft-lora-rlhf-skywork_results.zip` -> **Download**. Loyiha ildizida oching (ichidagi yo'llar `chat_ui/...`,
`rlhf/...`, `eval/...` dan boshlanadi), keyin `python eval/summarize.py`.